In [25]:
import os
print(os.getcwd())
print(os.listdir("."))

/Users/unipod/Downloads/project
['.DS_Store', 'loan_default.ipynb', 'data']


In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = "data/"
prev = pd.read_csv(DATA_DIR + "trainprevloans.csv")
perf = pd.read_csv(DATA_DIR + "trainperf.csv")
demo = pd.read_csv(DATA_DIR + "traindemographics.csv")
print(perf.shape, prev.shape, demo.shape)
print(perf["good_bad_flag"].value_counts(normalize=True))

(4368, 10) (18183, 12) (4346, 9)
good_bad_flag
Good    0.782051
Bad     0.217949
Name: proportion, dtype: float64


In [ ]:
prev['customerid'].nunique()

4359

In [ ]:
def check_unique(data, column='customerid'):
    n_rows = len(data)
    n_unique=data[column].nunique()
    return n_rows - n_unique
for name, df in [("perf", perf), ("prev", prev), ("demo", demo)]:
    print(name, check_unique(df))
    
    

perf 0
prev 13824
demo 12


In [ ]:
demo = demo.drop_duplicates("customerid")
print(demo.customerid.nunique(), len(demo))

4334 4334


In [ ]:
prev.head()

,customerid,systemloanid,loannumber,approveddate,creationdate,loanamount,totaldue,termdays,closeddate,referredby,firstduedate,firstrepaiddate
0,8a2a81a74ce8c05d014cfb32a0da1049,301682320,2,2016-08-15 18:22:40.000000,2016-08-15 17:22:32.000000,10000.0,13000.0,30,2016-09-01 16:06:48.000000,NaN,2016-09-14 00:00:00.000000,2016-09-01 15:51:43.000000
1,8a2a81a74ce8c05d014cfb32a0da1049,301883808,9,2017-04-28 18:39:07.000000,2017-04-28 17:38:53.000000,10000.0,13000.0,30,2017-05-28 14:44:49.000000,NaN,2017-05-30 00:00:00.000000,2017-05-26 00:00:00.000000
2,8a2a81a74ce8c05d014cfb32a0da1049,301831714,8,2017-03-05 10:56:25.000000,2017-03-05 09:56:19.000000,20000.0,23800.0,30,2017-04-26 22:18:56.000000,NaN,2017-04-04 00:00:00.000000,2017-04-26 22:03:47.000000
3,8a8588f35438fe12015444567666018e,301861541,5,2017-04-09 18:25:55.000000,2017-04-09 17:25:42.000000,10000.0,11500.0,15,2017-04-24 01:35:52.000000,NaN,2017-04-24 00:00:00.000000,2017-04-24 00:48:43.000000
4,8a85890754145ace015429211b513e16,301941754,2,2017-06-17 09:29:57.000000,2017-06-17 08:29:50.000000,10000.0,11500.0,15,2017-07-14 21:18:43.000000,NaN,2017-07-03 00:00:00.000000,2017-07-14 21:08:35.000000


In [ ]:
prev["firstduedate"] = pd.to_datetime(prev["firstduedate"])
prev["firstrepaiddate"] =pd.to_datetime(prev["firstrepaiddate"])
prev["days_late"] = (prev['firstrepaiddate'] - prev['firstduedate']).dt.days
prev['days_late'].describe()


count    18183.000000
mean        -2.473904
std          9.900832
min        -32.000000
25%         -6.000000
50%         -2.000000
75%          0.000000
max        351.000000
Name: days_late, dtype: float64

In [ ]:
prev["paid_late"] = (prev["days_late"] > 0).astype(int)

history = prev.groupby("customerid").agg(
    n_prev=("systemloanid", "count"),
    mean_late = ("days_late", "mean"),
    max_late=("days_late", "max"),
    late_share = ("paid_late", "mean"),

)
print(history.shape)
history.head()



(4359, 4)


,n_prev,mean_late,max_late,late_share
customerid,,,,
8a1088a0484472eb01484669e3ce4e0b,1,6.000000,6,1.000000
8a1a1e7e4f707f8b014f797718316cad,4,-0.250000,1,0.250000
8a1a32fc49b632520149c3b8fdf85139,7,-0.428571,1,0.142857
8a1eb5ba49a682300149c3c068b806c7,8,-3.125000,8,0.125000
8a1edbf14734127f0147356fdb1b1eb2,2,-4.000000,0,0.000000


In [ ]:
perf['interest_ratio'] = (perf['totaldue']) / (perf['loanamount'])
train = perf.merge(history,on='customerid', how='left'
    )
train = perf.merge(history, on="customerid", how="left")
train = train.merge(demo[['customerid', 'birthdate']], on='customerid', how='left')
train['age'] = 2017 - pd.to_datetime(train['birthdate']).dt.year

print(train.columns.tolist())
print(train['age'].describe())
print(len(train))

['customerid', 'systemloanid', 'loannumber', 'approveddate', 'creationdate', 'loanamount', 'totaldue', 'termdays', 'referredby', 'good_bad_flag', 'interest_ratio', 'n_prev', 'mean_late', 'max_late', 'late_share', 'birthdate', 'age']
count    3269.000000
mean       32.909758
std         6.132223
min        21.000000
25%        28.000000
50%        32.000000
75%        37.000000
max        56.000000
Name: age, dtype: float64
4368


In [ ]:
train

,customerid,systemloanid,loannumber,approveddate,creationdate,loanamount,totaldue,termdays,referredby,good_bad_flag,interest_ratio,n_prev,mean_late,max_late,late_share,birthdate,age
0,8a2a81a74ce8c05d014cfb32a0da1049,301994762,12,2017-07-25 08:22:56.000000,2017-07-25 07:22:47.000000,30000.0,34500.0,30,NaN,Good,1.1500,11.0,-0.909091,39.0,0.272727,1972-01-15 00:00:00.000000,45.0
1,8a85886e54beabf90154c0a29ae757c0,301965204,2,2017-07-05 17:04:41.000000,2017-07-05 16:04:18.000000,15000.0,17250.0,30,NaN,Good,1.1500,NaN,NaN,NaN,NaN,1985-08-23 00:00:00.000000,32.0
2,8a8588f35438fe12015444567666018e,301966580,7,2017-07-06 14:52:57.000000,2017-07-06 13:52:51.000000,20000.0,22250.0,15,NaN,Good,1.1125,6.0,0.833333,30.0,0.166667,1984-09-18 00:00:00.000000,33.0
3,8a85890754145ace015429211b513e16,301999343,3,2017-07-27 19:00:41.000000,2017-07-27 18:00:35.000000,10000.0,11500.0,15,NaN,Good,1.1500,2.0,7.500000,11.0,1.000000,1977-10-10 00:00:00.000000,40.0
4,8a858970548359cc0154883481981866,301962360,9,2017-07-03 23:42:45.000000,2017-07-03 22:42:39.000000,40000.0,44000.0,30,NaN,Good,1.1000,8.0,-3.125000,-1.0,0.000000,1986-09-07 00:00:00.000000,31.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4363,8a858e6d58b0cc520158beeb14b22a5a,302003163,2,2017-07-30 09:19:42.000000,2017-07-30 08:18:30.000000,10000.0,13000.0,30,NaN,Bad,1.3000,1.0,5.000000,5.0,1.000000,NaN,NaN
4364,8a858ee85cf400f5015cf44ab1c42d5c,301998967,2,2017-07-27 15:35:47.000000,2017-07-27 14:35:40.000000,10000.0,13000.0,30,NaN,Bad,1.3000,1.0,-4.000000,-4.0,0.000000,1989-12-04 00:00:00.000000,28.0
4365,8a858f365b2547f3015b284597147c94,301995576,3,2017-07-25 16:25:57.000000,2017-07-25 15:24:47.000000,10000.0,11500.0,15,NaN,Bad,1.1500,2.0,8.000000,16.0,0.500000,1989-11-21 00:00:00.000000,28.0
4366,8a858f935ca09667015ca0ee3bc63f51,301977679,2,2017-07-14 13:50:27.000000,2017-07-14 12:50:21.000000,10000.0,13000.0,30,8a858eda5c8863ff015c9dead65807bb,Bad,1.3000,1.0,0.000000,0.0,0.000000,1990-06-11 00:00:00.000000,27.0


Training the model

In [ ]:
y = (train["good_bad_flag"] == "Good").astype(int)
features = ["loanamount", "totaldue", "termdays", "interest_ratio", "n_prev", "mean_late", "max_late", "late_share", "age", "loannumber"]
X = train[features]

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score


model = HistGradientBoostingClassifier(random_state=0)
scores = cross_val_score (model, X, y, cv=5, scoring = 'accuracy')
print(scores)
print("error:", 1- scores.mean())



[0.77688787 0.77574371 0.79519451 0.77892325 0.7766323 ]
error: 0.21932367149758458


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, stratify=y, random_state=0)
model.fit(X_train, y_train)
imp = permutation_importance(model, X_test, y_test, n_repeats = 5, random_state = 0)

print(pd.Series(imp.importances_mean, index=features).sort_values(ascending=False))



mean_late         0.013004
late_share        0.010073
max_late          0.006593
loannumber        0.004212
interest_ratio    0.000366
loanamount        0.000183
age              -0.000183
termdays         -0.001099
totaldue         -0.004212
n_prev           -0.004396
dtype: float64


In [ ]:
train.groupby(train["age"].isna())["good_bad_flag"].value_counts(normalize=True)

age    good_bad_flag
False  Good             0.78189
       Bad              0.21811
True   Good             0.78253
       Bad              0.21747
Name: proportion, dtype: float64

Now let us train a model that is more consrain

In [ ]:
from sklearn.model_selection import cross_val_predict
y = (train["good_bad_flag"] == "Good").astype(int)
features = [ "interest_ratio",  "mean_late", "max_late", "late_share", "loannumber"]
X = train[features]

model = HistGradientBoostingClassifier( max_depth=3, learning_rate=0.05, max_iter=150,
                                       min_samples_leaf=50, l2_regularization=1.0, random_state=0)
proba = cross_val_predict(model, X,y, cv=5, method="predict_proba", )[:, 1]
for th in [0.4, 0.5, 0.5]:
    print(th, 1 - ((proba > th) == y).mean())


0.4 0.21199633699633702
0.5 0.20673076923076927
0.5 0.20673076923076927


Using the test set

In [ ]:
prev_t = pd.read_csv(DATA_DIR + "testprevloans.csv")
perf_t = pd.read_csv(DATA_DIR + "testperf.csv")

prev_t["firstduedate"] = pd.to_datetime(prev_t["firstduedate"])
prev_t["firstrepaiddate"] = pd.to_datetime(prev_t["firstrepaiddate"])
prev_t["days_late"] = (prev_t["firstrepaiddate"] - prev_t["firstduedate"]).dt.days
prev_t["paid_late"] = (prev_t["days_late"] > 0).astype(int)

history_t = prev_t.groupby("customerid").agg(
    n_prev=("systemloanid", "count"),
    mean_late=("days_late", "mean"),
    max_late=("days_late", "max"),
    late_share=("paid_late", "mean"),
)
print(len(history_t))


FileNotFoundError: [Errno 2] No such file or directory: 'datatestprevloans.csv'

In [ ]:
perf_t['interest_ratio'] = perf_t["totaldue"] / perf_t['loanamount']
test = perf_t.merge(history_t, on="customerid", how="left")
print(len(test))

In [ ]:
features = ["interest_ratio", "mean_late", "max_late", "late_share", "loannumber"]

model = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=150,
                                       min_samples_leaf=50, l2_regularization=1.0, random_state=0)
model.fit(train[features], y)

proba = model.predict_proba(test[features])[:, 1]
pred = (proba > 0.5).astype(int)

submission = pd.DataFrame({"customerid": test["customerid"], "Good_Bad_flag": pred})
submission.to_csv("submission.csv", index=False)
print(submission.shape)
print(submission["Good_Bad_flag"].value_counts(normalize=True))

In [29]:
!pip freeze | grep -iE "pandas|numpy|scikit-learn|matplotlib|seaborn"

matplotlib==3.9.4
matplotlib-inline==0.2.2
numpy==2.0.2
pandas==2.3.3
scikit-learn==1.6.1
seaborn==0.13.2
